In [1]:
import pandas as pd
import numpy as np

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
# Download walkscore values 
from bs4 import BeautifulSoup
from urllib.request import urlretrieve, urlopen
import xml.etree.ElementTree as ET
import pandas as pd
import requests

In [51]:
url = "https://www.walkscore.com/AU-VIC/Melbourne"

response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

walk_score = []
for q in soup.find_all('tr'):
    line = str(q.text)

    if line[1] == 'R':
        continue

    line = line.split("-")[0].split()[1:]
    score = int(line[-1])
    town = " ".join(line[:-1])
    walk_score.append([town, score])

walk_score_df = pd.DataFrame(walk_score, columns=['town', 'walkability'])
walk_score_df.head()

,town,walkability
0,Carlton,97
1,Fitzroy,96
2,Fitzroy North,93
3,Melbourne,93
4,St Kilda,93


In [52]:
data = pd.read_csv('../data/curated/merged_data.csv')
data['nbn'] = np.where(data['nbn'] == 'No NBN', 0, 1)
data['house'] = np.where(data['property type'] == 'House', 1, 0)
data['apartment'] = np.where(data['property type'] == 'Apartment / Unit / Flat', 1, 0)
data['townhouse'] = np.where(data['property type'] == 'Townhouse', 1, 0)
data['average offence count'] = (data['offence count 2023'] + data['offence count 2022'] + data['offence count 2021']) / 3
data['unemployed_percentage'] = data['unemployed (no.) 2021'] / data['employed (no.) 2021']

In [56]:
gb = data.groupby(['postcode']).agg(
    avg_beds=('beds', 'mean'),
    avg_baths=('baths', 'mean'),
    avg_parking=('baths', 'mean'),
    nbn_percent=('nbn', 'mean'),
    avg_rent=('weekly rent amount', 'mean'),
    median_rent=('weekly rent amount', 'median'),
    house_percent=('house', 'mean'),
    apartment_percent=('apartment', 'mean'),
    townhouse_percent=('townhouse', 'mean'),
    avg_prmy_school_dist=('distance primary school', 'mean'),
    avg_high_school_dist=('distance secondary school', 'mean'),
    avg_uni_dist=('distance university campus', 'mean'),
    offence_avg=('average offence count', 'mean'),
    unemployed_percent=('unemployed_percentage', 'mean'),
    labour_force=('in the labour force (no.) 2021', 'mean'),
    avg_families=('total families (no.) 2021', 'mean'),
    supermarkets=('total', 'mean'),
    town=('town', 'first')
)
gb = pd.merge(gb, walk_score_df, how='left', on='town')

# Now we want to normalise for labour force (i.e. those who are renting)

gb['offence_avg'] = gb['offence_avg'] / gb['labour_force']
gb['avg_families'] = gb['avg_families'] / gb['labour_force']
gb['supermarkets'] = gb['supermarkets'] / gb['labour_force']

# Also, we want to scale all towns to between 0 and 1. First we need to fill in NaN's in walkability with the mean
gb['walkability'].fillna('')

,avg_beds,avg_baths,avg_parking,nbn_percent,avg_rent,median_rent,house_percent,apartment_percent,townhouse_percent,avg_prmy_school_dist,avg_high_school_dist,avg_uni_dist,offence_avg,unemployed_percent,labour_force,avg_families,supermarkets,town,walkability
0,1.681511,1.311741,1.311741,0.614035,687.801619,630.0,0.016194,0.955466,0.000000,1.126371,0.516950,0.236295,119.666667,0.076478,60108.0,23110.0,11.0,Melbourne,93.0
1,1.793443,1.498361,1.498361,0.639344,742.901639,650.0,0.003279,0.996721,0.000000,0.955998,0.511133,0.732928,73.000000,0.057063,12899.0,4936.0,1.0,Southbank,91.0
2,1.686131,1.416058,1.416058,0.496350,707.058394,650.0,0.021898,0.970803,0.007299,0.833766,1.214619,2.876464,48.666667,0.050355,3545.0,1340.0,2.0,Docklands,86.0
3,2.159664,1.394958,1.394958,0.823529,545.647059,520.0,0.268908,0.621849,0.092437,0.404970,1.312890,0.809234,117.666667,0.076634,11448.0,5019.0,3.0,Footscray,78.0
4,2.418919,1.364865,1.364865,0.891892,510.067568,500.0,0.378378,0.391892,0.202703,0.693705,2.341281,3.984774,191.000000,0.084551,22117.0,11877.0,1.0,West Footscray,64.0
